# Portugal Rental Data Inspection

I restored this notebook inside `data_exploration` so the exploratory work stays near the data-prep scripts. I also updated it to match the current lean folder layout instead of the older broken raw-file paths.


## Setup Note

In this cell I import the small set of libraries I need and resolve paths so the notebook works whether I launch it from the repo root or from inside `data_exploration`.


In [ ]:
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
if (BASE_DIR / 'code files').exists():
    PROJECT_CODE_DIR = BASE_DIR / 'code files'
elif BASE_DIR.name == 'data_exploration':
    PROJECT_CODE_DIR = BASE_DIR.parent
else:
    PROJECT_CODE_DIR = BASE_DIR

DATA_EXPLORATION_DIR = PROJECT_CODE_DIR / 'data_exploration'
DATA_DIR = PROJECT_CODE_DIR / 'data'
PROJECT_CODE_DIR, DATA_EXPLORATION_DIR


## Load Project Modules Note

In this cell I import the local preprocessing module from the `data_exploration` folder so the inspection notebook stays aligned with the current project code.


In [ ]:
module_path = DATA_EXPLORATION_DIR / 'preprocessing.py'
spec = importlib.util.spec_from_file_location('preprocessing', module_path)
preprocessing = importlib.util.module_from_spec(spec)
spec.loader.exec_module(preprocessing)
module_path


## Load Current Datasets Note

In this cell I load the latest Airbnb raw listing snapshots for Lisbon and Porto together with the current Silver clean master listings file so I can inspect both the raw and cleaned layers side by side.


In [ ]:
lisbon_snapshot_dir = sorted((DATA_DIR / 'bronze' / 'raw' / 'Airbnb' / 'lisbon').iterdir())[-1]
porto_snapshot_dir = sorted((DATA_DIR / 'bronze' / 'raw' / 'Airbnb' / 'porto').iterdir())[-1]

lisbon_raw = pd.read_csv(lisbon_snapshot_dir / 'listings.csv.gz')
porto_raw = pd.read_csv(porto_snapshot_dir / 'listings.csv.gz')
clean_listings = pd.read_csv(DATA_DIR / 'silver' / 'listings' / 'clean_master_listings.csv')

lisbon_raw.shape, porto_raw.shape, clean_listings.shape


## Snapshot Preview Note

In this cell I look at a few rows from each layer so I can verify the fields and the current state of the dataset quickly.


In [ ]:
display(lisbon_raw.head(3))
display(porto_raw.head(3))
display(clean_listings.head(3))


## Coverage Summary Note

In this cell I summarize the current raw and clean coverage so I can see the listing volumes by source city and the basic pricing and occupancy distributions in the cleaned layer.


In [ ]:
summary = pd.DataFrame([
    {'dataset': 'lisbon_raw', 'rows': len(lisbon_raw), 'columns': lisbon_raw.shape[1]},
    {'dataset': 'porto_raw', 'rows': len(porto_raw), 'columns': porto_raw.shape[1]},
    {'dataset': 'clean_master_listings', 'rows': len(clean_listings), 'columns': clean_listings.shape[1]},
])
summary


## Clean Layer Distribution Note

In this cell I inspect a few core business variables from the Silver clean listings layer so I can understand price, occupancy, revenue, and city mix without having to jump into the ML notebooks yet.


In [ ]:
distribution_cols = [
    'city',
    'price',
    'occupancy_rate',
    'estimated_revenue_l365d',
    'room_type',
    'accommodates',
]
clean_listings[distribution_cols].describe(include='all').transpose()


## City Comparison Note

In this cell I create a lightweight city comparison table from the clean layer so I can quickly compare how Lisbon and Porto look in terms of listing count, median price, median occupancy, and median estimated annual revenue.


In [ ]:
city_compare = (
    clean_listings.groupby('city')
    .agg(
        listing_count=('property_id', 'count'),
        median_price=('price', 'median'),
        median_occupancy_rate=('occupancy_rate', 'median'),
        median_estimated_revenue=('estimated_revenue_l365d', 'median'),
    )
    .reset_index()
    .sort_values('listing_count', ascending=False)
)
city_compare
